In [1]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt > input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  4314k      0 --:--:-- --:--:-- --:--:-- 4322k


In [4]:
text = open("input.txt", 'r', encoding='utf-8').read()
text[:100], len(text)

('First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou',
 1115394)

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
stoi = {}
itos = {}
for i in range(vocab_size):
    stoi[chars[i]] = i
    itos[i] = chars[i]
encode = lambda s: [stoi[c] for c in s]
decode = lambda n: [itos[i] for i in n]
encoded = encode("hello world")
decoded = "".join(decode(encoded))
encoded, decoded

([46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42], 'hello world')

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
data = torch.tensor(encode(text), device=device)
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [9]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [ ]:
torch.manual_seed(1337)
block_size = 256
batch_size = 64
n_embd = 384
head_dim = 384
blocks = 6
n_heads = 6
dropout = 0.2
lr = 1e-3
steps = 2000


def get_batch(is_train=True):
    data = train_data if is_train else val_data
    idx = torch.randint(0, data.shape[0]-block_size, (batch_size, ))
    # we want shape batch x block
    x = torch.stack([data[i:i+block_size] for i in idx]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in idx]).to(device)
    return (x,y)
x,y = get_batch()
x.shape, y.shape

(torch.Size([64, 256]), torch.Size([64, 256]))

In [50]:
class Head(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        
        self.K = nn.Linear(n_embd, head_dim, bias=False)
        self.Q = nn.Linear(n_embd, head_dim, bias=False)
        self.V = nn.Linear(n_embd, head_dim, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))

    def forward(self, x):
        # x is B, T, C

        k = self.K(x) # B, T, head_dim
        q = self.Q(x)
        v = self.V(x)
        out = q @ k.transpose(-2,-1) * head_dim**0.5# B, T, T
        out = out.masked_fill(self.tril== 0, float('-inf'))
        out = F.softmax(out, -1)
        out = out @ v # B, T, head_dim
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, head_amt, head_dim):
        super().__init__()

        self.head_amt = head_amt
        self.head_dim = head_dim

        self.KQV = nn.Linear(n_embd, 3 * head_dim * head_amt)

        self.proj = nn.Linear(head_dim * head_amt, n_embd)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape
        N = self.head_amt
        D = self.head_dim

        # (B, T, 3*N*D)
        kqv = self.KQV(x)

        # Each (B, T, N*D)
        k, q, v = kqv.chunk(3, dim=-1)

        # Each (B, N, T, D)
        k = k.reshape(B, T, N, D).transpose(1, 2)
        q = q.reshape(B, T, N, D).transpose(1, 2)
        v = v.reshape(B, T, N, D).transpose(1, 2)

        # (B, N, T, T)
        w = q @ k.transpose(-2, -1) * D**-0.5

        # Causal masking
        w = w.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        w = F.softmax(w, dim=-1)

        # (B, N, T, D)
        out = w @ v

        # (B, T, N, D) -> (B, T, N*D)
        out = out.transpose(1, 2).contiguous()
        out = out.reshape(B, T, N * D)

        # (B, T, n_embd)
        out = self.proj(out)
        out = self.dropout(out)

        return out

class FeedForward(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_in, n_out*4),
            nn.ReLU(),
            nn.Linear(n_out*4, n_out),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.seq(x)

class Block(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.sa_heads = MultiHeadAttention(n_heads, n_embd//n_heads)
        self.ffwrd = FeedForward(n_embd//n_heads*n_heads, n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwrd(self.ln2(x))
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_table = nn.Embedding(block_size, n_embd)
        self.emb_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd) for _ in range(blocks)])
        self.lm = nn.Linear(n_embd, vocab_size)

    def forward(self, x, targets=None):
        pos = self.pos_table(torch.arange(block_size, device=device)) # block_size, n_embd
        x = pos + self.emb_table(x).to(device) # B, block_size, n_embd
        for block in self.blocks:
            x = block(x)
        x = self.lm(x) # B, T, vocab_size
        if targets == None:
            loss = None
        else:
            B, T, C = x.shape
            x = x.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(x, targets)
        return x, loss

    def generate(self, context, max_len):
        for _ in range(max_len):
            clipped = context[:,-block_size:]
            logits, _ = self(clipped) 
            probs = F.softmax(logits, -1)[:,-1,:].squeeze(-1)
            context = torch.cat((context, torch.multinomial(probs, num_samples=1)), dim=1)
        return context


In [51]:
m = Model()
m.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

In [21]:
for step in range(steps):
    x,y = get_batch()
    logits, loss = m(x,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        with torch.no_grad():
            m.eval()
            xtr,ytr = get_batch()
            _, tr_loss = m(xtr,ytr)
            xval,yval = get_batch(is_train=False)
            _, val_loss = m(xval, yval)
            print(f"At step {step}, train loss: {tr_loss}, val loss: {val_loss}")
            m.train()

At step 0, train loss: 3.116016387939453, val loss: 3.1722865104675293
At step 100, train loss: 2.5144171714782715, val loss: 2.537752628326416
At step 200, train loss: 2.4650232791900635, val loss: 2.493483304977417
At step 300, train loss: 2.4405908584594727, val loss: 2.4473555088043213
At step 400, train loss: 2.4143848419189453, val loss: 2.4435768127441406
At step 500, train loss: 2.376321315765381, val loss: 2.3950817584991455
At step 600, train loss: 2.292205810546875, val loss: 2.3176918029785156
At step 700, train loss: 2.1911263465881348, val loss: 2.2174205780029297
At step 800, train loss: 2.0825185775756836, val loss: 2.1445152759552
At step 900, train loss: 2.0480966567993164, val loss: 2.087029457092285
At step 1000, train loss: 1.9789185523986816, val loss: 2.050748825073242
At step 1100, train loss: 1.930711030960083, val loss: 1.9949994087219238
At step 1200, train loss: 1.8720871210098267, val loss: 1.9749462604522705
At step 1300, train loss: 1.8700660467147827, va

In [52]:
# torch.save(m.state_dict(), 'model_weights.pth')
m.load_state_dict(torch.load('model_weights.pth', weights_only=True))

<All keys matched successfully>

In [ ]:
out = m.generate(torch.randint(0,vocab_size, (block_size, ), device=device).unsqueeze(0), 500)
print("".join(decode(out.squeeze(0).tolist())))

RuntimeError: The size of tensor a (256) must match the size of tensor b (2) at non-singleton dimension 1